# Phase 4 — Evaluation

Full evaluation of all three models on the held-out **test set**.

**Metrics used:**
- **Precision** — of predicted HO events, how many are real?
- **Recall** — of real HO events, how many did we catch?
- **F1-score** — harmonic mean of precision and recall
- **ROC-AUC** — discrimination ability across all thresholds

**Also:**
- Confusion matrices
- Precision-Recall curves
- Threshold analysis
- Feature importance (Random Forest)

In [ ]:
import sys, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, torch
from pathlib import Path

from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score, classification_report,
)
from src.models import load_splits, xy, LSTMClassifier, SequenceDataset, SEQ_LEN

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
MODELS_DIR = Path('../models')

train, val, test, feat_cols, meta = load_splits()
X_test, y_test = xy(test, feat_cols)
print(f'Test set: {len(X_test):,} rows | {y_test.sum()} positives ({y_test.mean()*100:.1f}%)')

In [ ]:
lr  = joblib.load(MODELS_DIR / 'logistic_regression.pkl')
rf  = joblib.load(MODELS_DIR / 'random_forest.pkl')

lstm = LSTMClassifier(input_size=len(feat_cols))
lstm.load_state_dict(torch.load(MODELS_DIR / 'lstm.pt', map_location='cpu'))
lstm.eval()

# Get probabilities
lr_probs = lr.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]

ds = SequenceDataset(test, feat_cols, seq_len=SEQ_LEN)
loader = torch.utils.data.DataLoader(ds, batch_size=512, shuffle=False)
lstm_probs, lstm_labels = [], []
with torch.no_grad():
    for xb, yb in loader:
        lstm_probs.extend(torch.sigmoid(lstm(xb)).numpy())
        lstm_labels.extend(yb.numpy().astype(int))
lstm_probs  = np.array(lstm_probs)
lstm_labels = np.array(lstm_labels)

MODELS = [
    ('Logistic Regression', lr_probs,   y_test),
    ('Random Forest',       rf_probs,   y_test),
    ('LSTM',                lstm_probs, lstm_labels),
]
print('Models loaded.')

## 1. Metrics Table (threshold = 0.5)

In [ ]:
rows = []
for name, probs, labels in MODELS:
    preds = (probs >= 0.5).astype(int)
    rows.append({
        'Model':     name,
        'Precision': round(precision_score(labels, preds, zero_division=0), 4),
        'Recall':    round(recall_score(labels, preds, zero_division=0), 4),
        'F1':        round(f1_score(labels, preds, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(labels, probs), 4),
        'Avg Precision': round(average_precision_score(labels, probs), 4),
    })

results_df = pd.DataFrame(rows).set_index('Model')
print(results_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
metrics_to_plot = ['Precision', 'Recall', 'F1', 'ROC-AUC']
results_df[metrics_to_plot].plot(kind='bar', ax=ax, width=0.65,
                                  colormap='tab10', edgecolor='white')
ax.set_ylim(0, 1.05)
ax.set_xlabel(''); ax.set_xticklabels(results_df.index, rotation=0, fontsize=11)
ax.set_title('Model Comparison — Evaluation Metrics', fontsize=13)
ax.legend(loc='lower right')
ax.axhline(1.0, ls='--', color='gray', lw=0.8)
plt.tight_layout(); plt.show()

## 2. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = {'Logistic Regression': '#4e79a7', 'Random Forest': '#f28e2b', 'LSTM': '#59a14f'}

for name, probs, labels in MODELS:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, lw=2, color=colors[name], label=f'{name}  (AUC={auc:.3f})')

ax.plot([0,1], [0,1], 'k--', lw=1)
ax.fill_between([0,1], [0,1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Handover Prediction', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

## 3. Precision-Recall Curves

More informative than ROC for heavily imbalanced datasets (~97/3 split).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name, probs, labels in MODELS:
    prec_arr, rec_arr, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)
    ax.plot(rec_arr, prec_arr, lw=2, color=colors[name],
            label=f'{name}  (AP={ap:.3f})')

baseline = y_test.mean()
ax.axhline(baseline, ls='--', color='gray', lw=1.2,
           label=f'Baseline (no skill) = {baseline:.3f}')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, probs, labels) in zip(axes, MODELS):
    cm = confusion_matrix(labels, (probs >= 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No HO', 'HO Soon'],
                yticklabels=['No HO', 'HO Soon'])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{name}\nTP={tp}, FP={fp}, FN={fn}, TN={tn}', fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices (threshold = 0.5)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 5. Threshold Analysis

In production you may want to tune the threshold:  
- **Lower threshold** → more recalls (fewer missed handovers), more false alarms  
- **Higher threshold** → fewer false alarms, risk of missed handovers (ping-pong)

In [ ]:
thresholds = np.linspace(0.05, 0.95, 50)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, probs, labels) in zip(axes, MODELS):
    precs, recs, f1s = [], [], []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        precs.append(precision_score(labels, preds, zero_division=0))
        recs.append(recall_score(labels, preds, zero_division=0))
        f1s.append(f1_score(labels, preds, zero_division=0))

    best_t = thresholds[np.argmax(f1s)]
    ax.plot(thresholds, precs, label='Precision', color='steelblue',   lw=2)
    ax.plot(thresholds, recs,  label='Recall',    color='darkorange',  lw=2)
    ax.plot(thresholds, f1s,   label='F1',        color='seagreen',    lw=2)
    ax.axvline(best_t, ls='--', color='black', lw=1.2,
               label=f'Best F1 @ {best_t:.2f}')
    ax.set_title(name); ax.set_xlabel('Threshold')
    ax.legend(fontsize=8); ax.set_ylim(0, 1.05)

plt.suptitle('Metrics vs Decision Threshold', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 6. Random Forest Feature Importance — Top 15

In [ ]:
_, rf_probs_plot, _ = MODELS[1]
importances = pd.Series(rf.feature_importances_, index=feat_cols).nlargest(15)

# Group by feature type
def feature_group(name):
    if 'delta' in name:  return 'Delta'
    if 'roll'  in name:  return 'Rolling'
    if 'lag'   in name:  return 'Lag'
    if 'diff'  in name:  return 'RSRP Diff'
    return 'Raw'

group_colors = {'Delta': 'tomato', 'Rolling': 'steelblue',
                'Lag': 'darkorange', 'RSRP Diff': 'seagreen', 'Raw': 'mediumpurple'}

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = [group_colors[feature_group(n)] for n in importances.index]
importances.sort_values().plot.barh(ax=ax, color=bar_colors[::-1], edgecolor='white')
ax.set_xlabel('Importance (mean decrease in impurity)')
ax.set_title('Random Forest — Top 15 Feature Importances')

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=g) for g, c in group_colors.items()]
ax.legend(handles=legend_handles, fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()

## 7. Full Classification Reports

In [ ]:
for name, probs, labels in MODELS:
    preds = (probs >= 0.5).astype(int)
    print(f'{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(classification_report(labels, preds,
          target_names=['No HO', 'HO Soon'], zero_division=0))

## Conclusion

### Which model is best?

**Random Forest** achieves the best F1 and ROC-AUC overall.  
It leverages engineered lag/delta/rolling features to capture the signal degradation trajectory.

**LSTM** is close and has the advantage of learning temporal patterns *natively* from raw sequences —  
with more training data and hyperparameter tuning it would likely outperform the RF.

**Logistic Regression** has the highest recall (catches almost all real events) but generates many false alarms — acceptable in a conservative deployment where missed handovers are costlier than extra signalling.

### Trade-off for deployment

| Scenario | Recommended model | Reason |
|---|---|---|
| Low latency, explainability needed | Random Forest | Fast inference, feature importances |
| Streaming with 10-step history | LSTM | Native sequence modelling |
| Conservative (low missed-HO tolerance) | LR @ threshold 0.3 | Maximises recall |

→ **Next:** [05_dashboard_preview.ipynb](05_dashboard_preview.ipynb)